In [1]:
import pandas as pd
import ast

df = pd.read_csv('../../../data/preprocessed/steam_indie_games_graded.csv')

In [2]:
# 2. 공식 분류 필터 정의
sub_genres = ['Roguelike', 'Roguelite', 'Metroidvania', 'Souls-like', 'Dungeon Crawler', 'Bullet Hell', 'Hack and Slash', 'Platformer', 'Puzzle', 'Simulation', 'Strategy', 'RPG', 'Action', 'Adventure', 'Point & Click', 'Survival']
visual_dim = ['2D', '3D', '2.5D']
visual_per = ['First-Person', 'Third Person', 'Top-Down', 'Isometric', 'Side Scroller']
visual_sty = ['Pixel Graphics', 'Realistic', 'Anime', 'Cute', 'Minimalist', 'Hand-drawn', 'Voxel', 'Low-Poly', 'Cartoony', 'Stylized']
themes = ['Sci-fi', 'Fantasy', 'Space', 'Zombies', 'Post-apocalyptic', 'Cyberpunk', 'Historical', 'War', 'Mystery', 'Lovecraftian', 'Magic', 'Medieval', 'Military']
moods = ['Relaxing', 'Funny', 'Atmospheric', 'Psychological Horror', 'Dark', 'Difficult', 'Emotional', 'Story Rich', 'Surreal']
f_mechanics = ['Crafting', 'Resource Management', 'Tactical', 'Turn-Based', 'Real-Time', 'Choice Matters']
f_design = ['Procedural Generation', 'Sandbox', 'Physics', 'Open World']
f_activities = ['Exploration', 'Mining', 'Building', 'Trading']

In [3]:
def extract_tags(tag_str, target_list):
    try:
        tags_dict = ast.literal_eval(tag_str)
        return [t for t in tags_dict.keys() if t in target_list]
    except: return []

# 3. 공식 가이드라인 기반 9개 열 생성
df['sub_genres'] = df['tags'].apply(lambda x: extract_tags(x, sub_genres))
df['visual_dimensions'] = df['tags'].apply(lambda x: extract_tags(x, visual_dim))
df['visual_perspectives'] = df['tags'].apply(lambda x: extract_tags(x, visual_per))
df['visual_styles'] = df['tags'].apply(lambda x: extract_tags(x, visual_sty))
df['themes'] = df['tags'].apply(lambda x: extract_tags(x, themes))
df['moods'] = df['tags'].apply(lambda x: extract_tags(x, moods))
df['features_mechanics'] = df['tags'].apply(lambda x: extract_tags(x, f_mechanics))
df['features_design'] = df['tags'].apply(lambda x: extract_tags(x, f_design))
df['features_activities'] = df['tags'].apply(lambda x: extract_tags(x, f_activities))

In [16]:
# 4. 카테고리별 태그 개수 확인 및 선별
from collections import Counter

categories = ['sub_genres', 'visual_dimensions', 'visual_perspectives', 'visual_styles', 'themes', 'moods', 'features_mechanics', 'features_design']

tags_to_keep = []

for cat in categories:
    
    all_tags = []
    
    # 해당 열의 모든 행을 순회
    for item in df[cat]:
        all_tags.extend(item)   # 리스트의 내용물을 all_tags에 추가
    
    # 빈도 계산 및 정렬
    counts = Counter(all_tags)
    safe_tags = [tag for tag, count in counts.items() if count >= 100]
    tags_to_keep.extend(safe_tags)
    count_df = pd.DataFrame(counts.items(), columns=['Tag', 'Count']).sort_values(by='Count', ascending=False)
    
    
    print(f"\n=== {cat.upper()} (Unique Tags: {len(count_df)}) ===")
    print(count_df.reset_index(drop=True))


=== SUB_GENRES (Unique Tags: 14) ===
                Tag  Count
0         Adventure   4414
1            Action   3848
2        Simulation   2347
3            Puzzle   2160
4               RPG   2109
5          Strategy   1921
6        Platformer   1124
7          Survival    851
8     Point & Click    698
9       Bullet Hell    604
10   Hack and Slash    496
11  Dungeon Crawler    473
12     Metroidvania    265
13       Souls-like    264

=== VISUAL_DIMENSIONS (Unique Tags: 3) ===
    Tag  Count
0    2D   3837
1    3D   2981
2  2.5D    331

=== VISUAL_PERSPECTIVES (Unique Tags: 5) ===
             Tag  Count
0   First-Person   2121
1       Top-Down   1151
2   Third Person    868
3  Side Scroller    584
4      Isometric    375

=== VISUAL_STYLES (Unique Tags: 9) ===
              Tag  Count
0            Cute   2228
1  Pixel Graphics   2197
2        Stylized   1388
3       Realistic   1120
4           Anime   1014
5        Cartoony   1000
6      Hand-drawn    947
7      Minimalist    75

In [17]:
tags_to_keep

['RPG',
 'Puzzle',
 'Dungeon Crawler',
 'Adventure',
 'Point & Click',
 'Action',
 'Strategy',
 'Survival',
 'Simulation',
 'Hack and Slash',
 'Platformer',
 'Metroidvania',
 'Bullet Hell',
 'Souls-like',
 '2D',
 '3D',
 '2.5D',
 'Top-Down',
 'First-Person',
 'Side Scroller',
 'Isometric',
 'Third Person',
 'Cartoony',
 'Stylized',
 'Cute',
 'Anime',
 'Pixel Graphics',
 'Minimalist',
 'Hand-drawn',
 'Realistic',
 'Fantasy',
 'Mystery',
 'Lovecraftian',
 'Zombies',
 'Post-apocalyptic',
 'Sci-fi',
 'Magic',
 'Space',
 'Military',
 'Cyberpunk',
 'War',
 'Historical',
 'Medieval',
 'Difficult',
 'Dark',
 'Story Rich',
 'Atmospheric',
 'Psychological Horror',
 'Funny',
 'Relaxing',
 'Emotional',
 'Surreal',
 'Turn-Based',
 'Resource Management',
 'Tactical',
 'Crafting',
 'Procedural Generation',
 'Sandbox',
 'Open World',
 'Physics']

In [4]:
# 분류 열 목록
official_cols = [
    'sub_genres', 'visual_dimensions', 'visual_perspectives', 
    'visual_styles', 'themes', 'moods', 
    'features_mechanics', 'features_design', 'features_activities'
]

In [5]:
def clean_tags(tag_list):
    # 1. 문자열을 리스트로 변환
    if isinstance(tag_list, str):
        tag_list = ast.literal_eval(tag_list)
    
    # 2. 빈 리스트인 경우 None 또는 특정 값으로 반환
    return tag_list if len(tag_list) > 0 else None

for col in official_cols:
    df[col] = df[col].apply(clean_tags)

# 결과 저장
df.to_csv('../../../data/preprocessed/steam_indie_games_taxonomy_cleaned.csv', index=False)

In [6]:
df.head()

,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,...,grade_label,sub_genres,visual_dimensions,visual_perspectives,visual_styles,themes,moods,features_mechanics,features_design,features_activities
0,226620,1912,364,14.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",2276,Desktop Dungeons,QCF Design,2023-04-18,Each step into the unknown heals you and revea...,...,대흥행,"[RPG, Puzzle, Dungeon Crawler]",[2D],[Top-Down],None,[Fantasy],[Difficult],"[Turn-Based, Resource Management]",[Procedural Generation],None
1,251570,327889,42157,44.99,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",370046,7 Days to Die,The Fun Pimps,2024-07-25,7 Days to Die is an open-world game that is a ...,...,대흥행,"[Action, Strategy, Survival, Simulation]",None,None,[Voxel],"[Zombies, Post-apocalyptic]",None,None,"[Sandbox, Open World, Procedural Generation]","[Building, Exploration]"
2,252190,157,98,19.99,"['Indie', 'RPG', 'Strategy']",255,Defender's Quest 2: Mists of Ruin,"Level Up Labs, LLC",2025-01-30,Rise above the Mirk and protect your ship from...,...,외면,"[RPG, Strategy]",[2D],None,"[Cartoony, Stylized]","[Sci-fi, Post-apocalyptic]",None,[Tactical],None,None
3,269770,7388,882,14.99,"['Action', 'Adventure', 'Indie', 'RPG']",8270,Secrets of Grindea,Pixel Ferrets,2024-02-29,Secrets of Grindea is an old-school Action RPG...,...,대흥행,"[RPG, Action, Adventure, Hack and Slash]",[2D],[Top-Down],"[Cute, Anime, Pixel Graphics]",[Fantasy],"[Funny, Story Rich]",None,None,None
4,276870,193,107,14.99,"['Indie', 'Simulation', 'Strategy']",300,Dwelvers,Rasmus Ljunggren,2023-12-19,Dwelvers is a unique real-time strategy Dungeo...,...,외면,"[Strategy, Simulation]",[3D],[Top-Down],"[Voxel, Stylized]","[Magic, Fantasy]",[Dark],[Resource Management],[Sandbox],"[Trading, Building, Exploration]"


In [ ]:
# 2. 리스트 파싱 함수
def parse_list(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        return ast.literal_eval(x)
    except:
        return []

# 3. 분석 대상 8개 카테고리 정의 (활동 제외)
categories = [
    'official_sub_genres', 'visual_dimensions', 'visual_perspectives',
    'visual_styles', 'official_themes', 'official_moods',
    'features_mechanics', 'features_design'
]

In [ ]:
# 4. 연도별 통계 집계 및 순위 산출 함수
def get_category_stats(df, categories):
    df['release_year'] = pd.to_datetime(df['release_date']).dt.year
    
    all_stats = []
    for cat in categories:
        # 각 카테고리 내 태그별로 행 분리 (Explode)
        temp_df = df[['release_year', 'total_reviews', 'positive_rate', cat]].copy()
        temp_df[cat] = temp_df[cat].apply(parse_list)
        exploded = temp_df.explode(cat).dropna(subset=[cat])
        
        # 태그별, 연도별 집계 (중앙값, 표준편차, 게임 수)
        stats = exploded.groupby(['release_year', cat]).agg(
            median_reviews=('total_reviews', 'median'),
            median_positive_rate=('positive_rate', 'median'),
            std_reviews=('total_reviews', 'std'),
            game_count=('total_reviews', 'count')
        ).reset_index()
        
        # 해당 연도 내에서 리뷰 중앙값 기준 내림차순 순위 부여
        stats['rank'] = stats.groupby('release_year')['median_reviews'].rank(ascending=False, method='min')
        stats['category'] = cat
        stats = stats.rename(columns={cat: 'tag'})
        all_stats.append(stats)
    
    return pd.concat(all_stats)

In [ ]:
# 통계 실행
final_stats = get_category_stats(df, categories)

In [20]:
df['satisfaction_grade']

0       high
1       high
2       high
3        low
4       high
        ... 
9164    high
9165     mid
9166     mid
9167    high
9168    high
Name: satisfaction_grade, Length: 9169, dtype: str